## 🤝 7. Advanced Orchestration

**Goal**: Different orchestration patterns like Group Chat, Hand off

| Type | Description |
|---|---|
| GroupChat | Multiple specialized agents converse to solve complex tasks, moderated by an orchestrator |
| HandOff | Assemble agents in a mesh topology, allowing them to transfer control to each other based on the conversation context. |


In [9]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from utils.config import get_foundry_client, get_env

from agent_framework import (
    Agent,
    AgentResponse,
    AgentResponseUpdate,
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowRunState,
    tool,
)

from agent_framework.orchestrations import GroupChatBuilder, HandoffBuilder

from agent_framework_foundry import FoundryChatClient

client = get_foundry_client()
print("✅ Connected to Foundry")

✅ Connected to Foundry


---
### Part A — Group Chat Workflow 
Group Chat workflow is a workflow where multiple specialized agents (e.g., coding, research, writing) converse to solve complex tasks, moderated by an orchestrator

In [11]:
# ── A1. Define agents ────────────────────────────────────────────────────────
from agent_framework import Message

ORCHESTRATOR_AGENT_INSTRUCTIONS = """
You coordinate a team conversation to solve the user's task.

Guidelines:
- Start with Researcher to gather information
- Then have Writer synthesize the final answer
- Only finish after both have contributed meaningfully
"""

orchestrator_agent = Agent(
    name="Orchestrator",
    description="Coordinates multi-agent collaboration by selecting speakers",
    instructions=ORCHESTRATOR_AGENT_INSTRUCTIONS,
    client=client,
)

# Participant agents
researcher = Agent(
    name="Researcher",
    description="Collects relevant background information",
    instructions="Gather concise facts that help a teammate answer the question.",
    client=client,
)

writer = Agent(
    name="Writer",
    description="Synthesizes polished answers from gathered information",
    instructions="Compose clear and structured answers using any notes provided.",
    client=client,
)

workflow = (
    GroupChatBuilder(
        participants=[researcher, writer],
        termination_condition=lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 4,
        intermediate_output_from=[researcher, writer],
        orchestrator_agent=orchestrator_agent,
        #output_from=[orchestrator_agent]
    )
    # Set a hard termination condition: stop after 4 assistant messages
    # The agent orchestrator will intelligently decide when to end before this limit but just in case
    .with_termination_condition(lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 4)
    .build()
)

print("✅ Workflow built successfully")

✅ Workflow built successfully


In [12]:
from typing import cast

task = "What are the key benefits of using async/await in Python? Provide a concise summary."

print("\nStarting Group Chat with Agent-Based Manager...\n")
print(f"TASK: {task}\n")
print("=" * 80)

# Keep track of the last response to format output nicely in streaming mode
last_response_id: str | None = None
async for event in workflow.run(task, stream=True):
    if event.type in ("intermediate", "output"):
        data = event.data
        if isinstance(data, AgentResponseUpdate):
            rid = data.response_id
            if rid != last_response_id:
                if last_response_id is not None:
                    print("\n")
                print(f"{data.author_name}:", end=" ", flush=True)
                last_response_id = rid
            print(data.text, end="", flush=True)
        elif event.type == "output":
            # The output of the group chat workflow is a collection of chat messages from all participants
            outputs = cast(list[Message], event.data)
            print("\n" + "=" * 80)
            print("\nFinal Conversation Transcript:\n")
            for message in outputs:
                print(f"{message.author_name or message.role}: {message.text}\n")


Starting Group Chat with Agent-Based Manager...

TASK: What are the key benefits of using async/await in Python? Provide a concise summary.

Researcher: 

Researcher: 1. **Improved Readability**: `async/await` syntax simplifies asynchronous code, making it look more like synchronous code.

2. **Efficient I/O Operations**: It allows for non-blocking I/O, enabling the program to handle other tasks while waiting for I/O operations to complete.

3. **Better Performance**: Reduces the need for threading, leading to lower overhead and better use of resources for I/O-bound tasks.

4. **Error Handling**: Easier to manage errors and cancellations in asynchronous code compared to callback-based solutions.

5. **Concurrency**: Facilitates concurrent task execution, improving the responsiveness and throughput of applications.

6. **Integration**: Naturally integrates with asyncio, Python's built-in library for writing concurrent code with coroutines. 

These benefits make `async/await` particular

---
### Part B — Handoff
A handoff workflow defines a pattern that assembles agents in a mesh topology, allowing
them to transfer control to each other based on the conversation context.

In [13]:
from typing import Annotated, cast

from agent_framework_orchestrations import HandoffAgentUserRequest

@tool(approval_mode="never_require")
def process_refund(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund for a given order number."""
    return f"Refund processed successfully for order {order_number}."


@tool(approval_mode="never_require")
def check_order_status(order_number: Annotated[str, "Order number to check status for"]) -> str:
    """Simulated function to check the status of a given order number."""
    return f"Order {order_number} is currently being processed and will ship in 2 business days."


@tool(approval_mode="never_require")
def process_return(order_number: Annotated[str, "Order number to process return for"]) -> str:
    """Simulated function to process a return for a given order number."""
    return f"Return initiated successfully for order {order_number}. You will receive return instructions via email."


def create_agents(client: FoundryChatClient) -> tuple[Agent, Agent, Agent, Agent]:
    """Create and configure the triage and specialist agents.

    Args:
        client: The FoundryChatClient to use for creating agents.

    Returns:
        Tuple of (triage_agent, refund_agent, order_agent, return_agent)
    """
    # Triage agent: Acts as the frontline dispatcher
    triage_agent = Agent(
        client=client,
        instructions=(
            "You are frontline support triage. Route customer issues to the appropriate specialist agents "
            "based on the problem described."
        ),
        name="triage_agent",
        require_per_service_call_history_persistence=True,
    )

    # Refund specialist: Handles refund requests
    refund_agent = Agent(
        client=client,
        instructions="You process refund requests.",
        name="refund_agent",
        # In a real application, an agent can have multiple tools; here we keep it simple
        tools=[process_refund],
        require_per_service_call_history_persistence=True,
    )

    # Order/shipping specialist: Resolves delivery issues
    order_agent = Agent(
        client=client,
        instructions="You handle order and shipping inquiries.",
        name="order_agent",
        # In a real application, an agent can have multiple tools; here we keep it simple
        tools=[check_order_status],
        require_per_service_call_history_persistence=True,
    )

    # Return specialist: Handles return requests
    return_agent = Agent(
        client=client,
        instructions="You manage product return requests.",
        name="return_agent",
        # In a real application, an agent can have multiple tools; here we keep it simple
        tools=[process_return],
        require_per_service_call_history_persistence=True,
    )

    return triage_agent, refund_agent, order_agent, return_agent


def _handle_events(events: list[WorkflowEvent]) -> list[WorkflowEvent[HandoffAgentUserRequest]]:
    """Process workflow events and extract any pending user input requests.

    This function inspects each event type and:
    - Prints workflow status changes (IDLE, IDLE_WITH_PENDING_REQUESTS, etc.)
    - Displays final conversation snapshots when workflow completes
    - Prints user input request prompts
    - Collects all request_info events for response handling

    Args:
        events: List of WorkflowEvent to process

    Returns:
        List of WorkflowEvent[HandoffAgentUserRequest] representing pending user input requests
    """
    requests: list[WorkflowEvent[HandoffAgentUserRequest]] = []

    for event in events:
        if event.type == "handoff_sent":
            # handoff_sent event: Indicates a handoff has been initiated
            print(f"\n[Handoff from {event.data.source} to {event.data.target} initiated.]")
        elif event.type == "status" and event.state in {
            WorkflowRunState.IDLE,
            WorkflowRunState.IDLE_WITH_PENDING_REQUESTS,
        }:
            # Status event: Indicates workflow state changes
            print(f"\n[Workflow Status] {event.state}")
        elif event.type == "output":
            # Output event: Contains contents generated by the workflow
            data = event.data
            if isinstance(data, AgentResponse):
                for message in data.messages:
                    if not message.text:
                        # Skip messages without text (e.g., tool calls)
                        continue
                    speaker = message.author_name or message.role
                    print(f"- {speaker}: {message.text}")
            elif event.type == "output":
                # The output of the handoff workflow is a collection of chat messages from all participants
                conversation = cast(list[Message], event.data)
                if isinstance(conversation, list):
                    print("\n=== Final Conversation Snapshot ===")
                    for message in conversation:
                        speaker = message.author_name or message.role
                        print(f"- {speaker}: {message.text or [content.type for content in message.contents]}")
                    print("===================================")
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            _print_handoff_agent_user_request(event.data.agent_response)
            requests.append(cast(WorkflowEvent[HandoffAgentUserRequest], event))

    return requests


def _print_handoff_agent_user_request(response: AgentResponse) -> None:
    """Display the agent's response messages when requesting user input.

    This will happen when an agent generates a response that doesn't trigger
    a handoff, i.e., the agent is asking the user for more information.

    Args:
        response: The AgentResponse from the agent requesting user input
    """
    if not response.messages:
        raise RuntimeError("Cannot print agent responses: response has no messages.")

    print("\n[Agent is requesting your input...]")

    # Print agent responses
    for message in response.messages:
        if not message.text:
            # Skip messages without text (e.g., tool calls)
            continue
        speaker = message.author_name or message.role
        print(f"- {speaker}: {message.text}")

In [14]:
# Create the agents for handsoff orchestration - triage + specialists
triage, refund, order, support = create_agents(client)

# Create the handoff workflow with custom termination condition
workflow = (
        HandoffBuilder(
            name="customer_support_handoff",
            participants=[triage, refund, order, support],
            # Custom termination: Check if one of the agents has provided a closing message.
            # This looks for the last message containing "welcome", which indicates the
            # conversation has concluded naturally.
            termination_condition=lambda conversation: (
                len(conversation) > 0 and "welcome" in conversation[-1].text.lower()
            ),
        )
        .with_start_agent(triage)
        .build()
    )

In [15]:
# Run the workflow with an initial user message and handle events
scripted_responses = [
    "My order 1234 arrived damaged and the packaging was destroyed. I'd like to return it.",
    "Please also process a refund for order 1234.",
    "What is the status of my order 3214?",
    "Thanks for resolving this.",
]

# Start the workflow with the initial user message
# run(..., stream=True) returns an async iterator of WorkflowEvent
print("[Starting workflow with initial user message...]\n")
initial_message = "Hello, I need assistance with my recent purchase."
print(f"- User: {initial_message}")

# Run the workflow and handle events until pending user input requests
workflow_result = workflow.run(initial_message, stream=True)
pending_requests = _handle_events([event async for event in workflow_result])

# Process the request/response cycle
# The workflow will continue requesting input until:
# 1. The termination condition is met, OR
# 2. We run out of scripted responses
while pending_requests:
    if not scripted_responses:
        # No more scripted responses; terminate the workflow
        responses = {req.request_id: HandoffAgentUserRequest.terminate() for req in pending_requests}
    else:
        # Get the next scripted response
        user_response = scripted_responses.pop(0)
        print(f"\n- User: {user_response}")

        # Send response(s) to all pending requests        
        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response) for req in pending_requests
        }

    # Send responses and get new events
    # We use run(responses=...) to get events from the workflow, allowing us to
    # display agent responses and handle new requests as they arrive
    events = await workflow.run(responses=responses)
    pending_requests = _handle_events(events)

[Starting workflow with initial user message...]

- User: Hello, I need assistance with my recent purchase.

[Agent is requesting your input...]
- triage_agent: I can help with that! Could you tell me more about the issue you're experiencing with your purchase? For example, is it about returning the item, getting a refund, or tracking your order?

[Workflow Status] WorkflowRunState.IDLE_WITH_PENDING_REQUESTS

- User: My order 1234 arrived damaged and the packaging was destroyed. I'd like to return it.

[Handoff from triage_agent to return_agent initiated.]
- return_agent: The return for order 1234 has been successfully initiated. You'll receive instructions via email on how to proceed with the return. If you have any other questions or need further assistance, feel free to ask!

[Agent is requesting your input...]
- return_agent: The return for order 1234 has been successfully initiated. You'll receive instructions via email on how to proceed with the return. If you have any other ques